[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/piertotumlocomotor/robot_balance/blob/develop/aprendizaje_refuerzo/guia_estudio.ipynb)

# Guía de estudio — cómo aprende el robot a balancearse

Versión para **entender**, no para entregar. Cada sección es corta y tiene una celda para tocar.
La primera celda prepara los archivos (no hace falta leerla).

In [ ]:
%pip install -q gymnasium

In [ ]:
#@title Preparar entorno.py { display-mode: "form" }
open('entorno.py', 'w').write('"""\nEntorno de simulación del robot balanceador (planta nueva: 2× BTS7960 + chasis de balsa).\n\nModelo: péndulo invertido linealizado de 1 grado de libertad.\n\n    J·θ̈ = m·g·l·θ + τ_motor(u)      →      θ̈ = ω₀²·θ + K_U·k(|u|)·u\n\n    θ    inclinación del cuerpo [rad], 0 = vertical\n    u    duty de PWM común a los dos motores (−121 … +121)\n    k()  fracción del torque disponible según la zona muerta de cada motor\n\nLo que el modelo NO incluye: posición/velocidad de las ruedas, fuerza\ncontraelectromotriz, juego de la reductora, dinámica eléctrica del motor.\n"""\n\nimport gymnasium as gym\nimport numpy as np\nfrom gymnasium import spaces\n\n# ---------------------------------------------------------------------------\n# 1. Parámetros físicos\n# ---------------------------------------------------------------------------\n# Medidos en la planta nueva (registro de pruebas §7.22 y §7.23)\nOMEGA0 = 8.0            # rad/s   frecuencia natural (video del robot colgado)\nJ = 0.0159              # kg·m²   momento de inercia respecto del eje\nM_L = 0.1040            # kg·m    masa × distancia al centro de masa (dos apoyos)\n\n# NO medido: el torque del motor con el driver nuevo (el ESP32 se quemó antes).\n# Se parte del torque medido con balanza en la planta vieja, dos motores en\n# brake a la vez (§7.20): τ = 0.0443 + 0.0339 N·m a duty 160 → 4.89e-4 N·m/duty.\n# El BTS7960 (MOSFET) no tiene la caída de ~3 V del L298N, así que entrega más\n# torque al mismo duty: entre 1.0× (pesimista) y 1.6× (optimista).\n# SUPUESTO del proyecto: 1.4×, medianamente optimista.\nFACTOR_DRIVER = 1.4\nTORQUE_POR_DUTY = FACTOR_DRIVER * (0.0443 + 0.0339) / 160      # N·m por unidad de duty\nK_U = TORQUE_POR_DUTY / J                                      # ≈ 0.043 rad/s² por unidad de duty\n\n# Zona muerta: duty por debajo del cual el motor no hace fuerza. Medida en la\n# planta vieja (92 y 102); como es un umbral de tensión, se escala con el driver.\nZONA_MUERTA_M1 = 92 / FACTOR_DRIVER     # ≈ 66\nZONA_MUERTA_M2 = 102 / FACTOR_DRIVER    # ≈ 73\nPROP_M1 = 0.0443 / (0.0443 + 0.0339)    # M1 aporta el 57% del torque (M2 es más débil)\nCAP = 121               # duty máximo: 6 V del motor / 12.6 V de batería llena (§8.5)\n\n# Ángulo máximo recuperable con el torque disponible: θ_umbral = K_U·CAP/ω₀² ≈ 4.7°\nTHETA_UMBRAL = K_U * CAP / OMEGA0**2\n\n# ---------------------------------------------------------------------------\n# 2. Definición del MDP\n# ---------------------------------------------------------------------------\nDT = 0.01               # paso de control: 100 Hz\nSUBPASOS = 10           # la física se integra en pasos de 1 ms\nDURACION = 5.0          # s por episodio → 500 pasos\nTHETA_CAIDA = np.radians(10.0)      # más inclinado que esto = robot caído\nTHETA0_MAX = np.radians(3.06)       # error de colocación a mano medido (Test 10)\nTHETA_DOT0_MAX = 0.15               # rad/s, ídem\n\n# Acciones: 9 niveles de duty. Todos los no nulos superan la zona muerta.\nNIVELES_DUTY = np.array([-121, -105, -90, -75, 0, 75, 90, 105, 121], dtype=np.float64)\n\n\nclass RobotBalanceadorEnv(gym.Env):\n    """\n    Estado observado (normalizado):  [θ/θ_caída,  θ̇/(ω₀·θ_caída),  u_anterior/CAP]\n    Acción:      índice 0..8 de NIVELES_DUTY\n    Recompensa:  1 − (θ/θ_caída)² − 0.05·(u/CAP)² − 0.05·(Δu/2CAP)²   por paso; 0 si cae\n    Fin:         terminated si |θ| > 10°;  truncated a los 5 s\n    """\n\n    def __init__(self, peso_u=0.05, peso_du=0.05):\n        super().__init__()\n        self.peso_u, self.peso_du = peso_u, peso_du\n        self.niveles = NIVELES_DUTY / CAP                 # en fracción del cap\n        self.max_pasos = int(round(DURACION / DT))\n        self.action_space = spaces.Discrete(len(NIVELES_DUTY))\n        self.observation_space = spaces.Box(-10.0, 10.0, shape=(3,), dtype=np.float32)\n\n    # -- física ------------------------------------------------------------\n    @staticmethod\n    def torque_disponible(u_abs):\n        """Fracción de K_U según cuántos motores superaron su zona muerta."""\n        if u_abs < ZONA_MUERTA_M1:\n            return 0.0\n        if u_abs < ZONA_MUERTA_M2:\n            return PROP_M1\n        return 1.0\n\n    def _integrar(self, u):\n        aceleracion_motor = K_U * self.torque_disponible(abs(u)) * u\n        dt = DT / SUBPASOS\n        for _ in range(SUBPASOS):                     # Euler semi-implícito\n            theta_ddot = OMEGA0**2 * self.theta + aceleracion_motor\n            self.theta_dot += theta_ddot * dt\n            self.theta += self.theta_dot * dt\n\n    @staticmethod\n    def es_recuperable(theta, theta_dot):\n        """El modo inestable del péndulo es x = θ + θ̇/ω₀. Con el torque máximo\n        solo se puede frenar si |x| < θ_umbral (cota optimista: ignora la zona muerta)."""\n        return abs(theta + theta_dot / OMEGA0) < THETA_UMBRAL\n\n    # -- interfaz Gymnasium -------------------------------------------------\n    def _obs(self):\n        return np.clip(np.array([self.theta / THETA_CAIDA,\n                                 self.theta_dot / (OMEGA0 * THETA_CAIDA),\n                                 self.u_anterior / CAP], dtype=np.float32), -10, 10)\n\n    def reset(self, *, seed=None, options=None):\n        super().reset(seed=seed)\n        options = options or {}\n        if "theta0" in options:                       # condición inicial fija (para graficar)\n            self.theta = float(options["theta0"])\n            self.theta_dot = float(options.get("theta_dot0", 0.0))\n        else:                                         # sorteo: como colocarlo a mano\n            while True:\n                self.theta = self.np_random.uniform(-THETA0_MAX, THETA0_MAX)\n                self.theta_dot = self.np_random.uniform(-THETA_DOT0_MAX, THETA_DOT0_MAX)\n                if self.es_recuperable(self.theta, self.theta_dot):\n                    break\n        self.paso, self.u_anterior = 0, 0.0\n        return self._obs(), {}\n\n    def step(self, accion):\n        u = NIVELES_DUTY[int(accion)]\n        self._integrar(u)\n        self.paso += 1\n\n        recompensa = (1.0\n                      - (self.theta / THETA_CAIDA) ** 2                       # estar vertical\n                      - self.peso_u * (u / CAP) ** 2                          # gastar poca energía\n                      - self.peso_du * ((u - self.u_anterior) / (2 * CAP)) ** 2)  # no "vibrar"\n        self.u_anterior = u\n\n        terminated = bool(abs(self.theta) > THETA_CAIDA)\n        if terminated:\n            recompensa = 0.0\n        truncated = self.paso >= self.max_pasos\n        info = {"theta": self.theta, "theta_dot": self.theta_dot, "u": u}\n        return self._obs(), float(recompensa), terminated, truncated, info\n')
print('escrito', 'entorno.py')

In [ ]:
#@title Preparar agentes.py { display-mode: "form" }
open('agentes.py', 'w').write('"""\nAgentes para el robot balanceador — Aprendizaje por Refuerzo I, Desafío 1.\n\n    DQN / Double DQN   basado en valor, off-policy (replay buffer + red objetivo)\n    A2C                actor-crítico, on-policy (n-step returns + GAE, entornos en paralelo)\n\nTodos usan exactamente el mismo MDP (entorno.py): la comparación es entre\nalgoritmos, no entre problemas.\n\nEstructura de DQN igual a la del TP3 / clase 5; la de A2C igual a la práctica\nde la clase 7 (RMSprop, init ortogonal, tanh, GAE opcional).\n"""\n\nimport random\nimport time\nfrom dataclasses import dataclass, field\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.distributions import Categorical\n\ndevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef fijar_semilla(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\ndef media_movil(x, w=50):\n    x = np.asarray(x, dtype=float)\n    w = max(1, min(w, len(x)))\n    return np.convolve(x, np.ones(w) / w, mode="valid")\n\n\n# ===========================================================================\n# DQN y Double DQN\n# ===========================================================================\n\n@dataclass\nclass HiperDQN:\n    pasos: int = 150_000          # pasos de entorno (cada paso = 10 ms simulados)\n    gamma: float = 0.99           # horizonte efectivo ~1/(1-γ) = 100 pasos = 1 s\n    lr: float = 5e-4\n    batch: int = 64\n    buffer: int = 50_000\n    inicio_aprendizaje: int = 2_000\n    copiar_objetivo_cada: int = 1_000\n    eps_inicial: float = 1.0\n    eps_final: float = 0.02\n    eps_pasos_decaimiento: int = 50_000\n    oculta: int = 64\n    double: bool = False\n\n\nclass ReplayBuffer:\n    def __init__(self, capacidad, n_obs):\n        self.capacidad, self.pos, self.tam = capacidad, 0, 0\n        self.s = np.zeros((capacidad, n_obs), dtype=np.float32)\n        self.s2 = np.zeros((capacidad, n_obs), dtype=np.float32)\n        self.a = np.zeros(capacidad, dtype=np.int64)\n        self.r = np.zeros(capacidad, dtype=np.float32)\n        self.d = np.zeros(capacidad, dtype=np.float32)\n\n    def guardar(self, s, a, r, s2, terminal):\n        i = self.pos\n        self.s[i], self.a[i], self.r[i], self.s2[i], self.d[i] = s, a, r, s2, terminal\n        self.pos = (self.pos + 1) % self.capacidad\n        self.tam = min(self.tam + 1, self.capacidad)\n\n    def muestrear(self, n):\n        idx = np.random.randint(0, self.tam, size=n)\n        t = lambda x: torch.as_tensor(x[idx], device=device)\n        return t(self.s), t(self.a).unsqueeze(1), t(self.r).unsqueeze(1), t(self.s2), t(self.d).unsqueeze(1)\n\n    def __len__(self):\n        return self.tam\n\n\nclass QNet(nn.Module):\n    """MLP: estado (3) → Q(s, a) para cada acción."""\n    def __init__(self, n_obs, n_acciones, oculta=64):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(n_obs, oculta), nn.ReLU(),\n            nn.Linear(oculta, oculta), nn.ReLU(),\n            nn.Linear(oculta, n_acciones),\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\ndef objetivo_td(red, red_objetivo, r, s2, d, gamma, double):\n    """y = r + γ·(1-terminal)·Q_objetivo(s\', a*)\n       DQN:        a* = argmax_a Q_objetivo(s\', a)   (la misma red elige y evalúa)\n       Double DQN: a* = argmax_a Q_red(s\', a)        (una elige, la otra evalúa)"""\n    with torch.no_grad():\n        if double:\n            a2 = red(s2).argmax(dim=1, keepdim=True)\n            q_sig = red_objetivo(s2).gather(1, a2)\n        else:\n            q_sig = red_objetivo(s2).max(dim=1, keepdim=True)[0]\n        return r + gamma * q_sig * (1.0 - d)\n\n\ndef entrenar_dqn(crear_env, hp: HiperDQN, seed=0, verbose=True):\n    """Devuelve (red, pasos_fin_episodio, retornos_por_episodio)."""\n    fijar_semilla(seed)\n    env = crear_env()\n    n_obs, n_acc = env.observation_space.shape[0], env.action_space.n\n    red = QNet(n_obs, n_acc, hp.oculta).to(device)\n    red_obj = QNet(n_obs, n_acc, hp.oculta).to(device)\n    red_obj.load_state_dict(red.state_dict())\n    opt = torch.optim.Adam(red.parameters(), lr=hp.lr)\n    buffer = ReplayBuffer(hp.buffer, n_obs)\n\n    s, _ = env.reset(seed=seed)\n    ret_ep, pasos_fin, retornos = 0.0, [], []\n    t0 = time.time()\n    for paso in range(1, hp.pasos + 1):\n        eps = max(hp.eps_final, hp.eps_inicial - (hp.eps_inicial - hp.eps_final) * paso / hp.eps_pasos_decaimiento)\n        if random.random() < eps:\n            a = random.randrange(n_acc)\n        else:\n            with torch.no_grad():\n                a = int(red(torch.as_tensor(s, device=device).unsqueeze(0)).argmax(dim=1).item())\n\n        s2, r, terminated, truncated, _ = env.step(a)\n        # Solo \'terminated\' (robot caído) corta el bootstrap: el truncamiento a\n        # los 5 s es un límite del episodio, no un estado terminal del robot.\n        buffer.guardar(s, a, r, s2, float(terminated))\n        s, ret_ep = s2, ret_ep + r\n\n        if len(buffer) >= max(hp.batch, hp.inicio_aprendizaje):\n            bs, ba, br, bs2, bd = buffer.muestrear(hp.batch)\n            q_sa = red(bs).gather(1, ba)\n            y = objetivo_td(red, red_obj, br, bs2, bd, hp.gamma, hp.double)\n            loss = F.smooth_l1_loss(q_sa, y)             # Huber\n            opt.zero_grad()\n            loss.backward()\n            nn.utils.clip_grad_norm_(red.parameters(), 10.0)\n            opt.step()\n\n        if paso % hp.copiar_objetivo_cada == 0:\n            red_obj.load_state_dict(red.state_dict())\n\n        if terminated or truncated:\n            pasos_fin.append(paso)\n            retornos.append(ret_ep)\n            if verbose and len(retornos) % 100 == 0:\n                print(f"  paso {paso:7d} | ε={eps:.3f} | ep {len(retornos):4d} | "\n                      f"media últ. 50: {np.mean(retornos[-50:]):6.1f}")\n            ret_ep = 0.0\n            s, _ = env.reset()\n    if verbose:\n        print(f"  {\'Double DQN\' if hp.double else \'DQN\'} seed {seed}: {(time.time() - t0) / 60:.1f} min")\n    return red, np.array(pasos_fin), np.array(retornos)\n\n\ndef politica_dqn(red):\n    def pi(obs):\n        with torch.no_grad():\n            return int(red(torch.as_tensor(obs, device=device).unsqueeze(0)).argmax(dim=1).item())\n    return pi\n\n\n# ===========================================================================\n# A2C (Advantage Actor-Critic)\n# ===========================================================================\n\n@dataclass\nclass HiperA2C:\n    pasos: int = 300_000          # pasos de entorno totales (sumando los N entornos)\n    n_envs: int = 8               # entornos en paralelo: decorrelacionan el batch (sin replay)\n    n_steps: int = 16             # largo del rollout antes de cada actualización\n    gamma: float = 0.99\n    gae_lambda: float = 0.95      # 1.0 = n-step puro; <1 baja varianza a cambio de sesgo\n    lr: float = 7e-4\n    ent_coef: float = 0.01        # bonus de entropía: mantiene exploración\n    vf_coef: float = 0.5\n    max_grad_norm: float = 0.5\n    oculta: int = 64\n    capas: int = 2\n\n\ndef _mlp(entrada, salida, oculta, capas, ganancia_salida):\n    mods, d = [], entrada\n    for _ in range(capas):\n        lin = nn.Linear(d, oculta)\n        nn.init.orthogonal_(lin.weight, np.sqrt(2)); nn.init.zeros_(lin.bias)\n        mods += [lin, nn.Tanh()]\n        d = oculta\n    ult = nn.Linear(d, salida)\n    nn.init.orthogonal_(ult.weight, ganancia_salida); nn.init.zeros_(ult.bias)\n    return nn.Sequential(*mods, ult)\n\n\nclass ActorCritico(nn.Module):\n    """Dos redes separadas: el actor da logits de π(a|s), el crítico estima V(s)."""\n    def __init__(self, n_obs, n_acciones, oculta=64, capas=2):\n        super().__init__()\n        self.actor = _mlp(n_obs, n_acciones, oculta, capas, 0.01)   # política inicial ~uniforme\n        self.critico = _mlp(n_obs, 1, oculta, capas, 1.0)\n\n    def dist(self, x):\n        return Categorical(logits=self.actor(x))\n\n    def valor(self, x):\n        return self.critico(x).squeeze(-1)\n\n\ndef ventajas_gae(r, v, v_ult, no_terminal, gamma, lam):\n    """GAE(λ) hacia atrás sobre un rollout [T, N].\n       δ_t = r_t + γ·V(s_{t+1})·(1-terminal) − V(s_t)\n       A_t = δ_t + γλ·(1-fin_t)·A_{t+1}\n    Con λ=1 es el retorno de n pasos menos V(s_t)."""\n    T = r.shape[0]\n    adv = torch.zeros_like(r)\n    ultimo = torch.zeros_like(v_ult)\n    for t in reversed(range(T)):\n        v_sig = v_ult if t == T - 1 else v[t + 1]\n        delta = r[t] + gamma * v_sig * no_terminal[t] - v[t]\n        ultimo = delta + gamma * lam * no_terminal[t] * ultimo\n        adv[t] = ultimo\n    return adv, adv + v\n\n\ndef entrenar_a2c(crear_env, hp: HiperA2C, seed=0, verbose=True):\n    """Devuelve (modelo, pasos_fin_episodio, retornos_por_episodio)."""\n    fijar_semilla(seed)\n    envs = [crear_env() for _ in range(hp.n_envs)]\n    n_obs, n_acc = envs[0].observation_space.shape[0], envs[0].action_space.n\n    modelo = ActorCritico(n_obs, n_acc, hp.oculta, hp.capas).to(device)\n    opt = torch.optim.RMSprop(modelo.parameters(), lr=hp.lr, alpha=0.99, eps=1e-5)\n\n    obs = np.stack([e.reset(seed=seed * 1000 + i)[0] for i, e in enumerate(envs)])\n    ret_ep = np.zeros(hp.n_envs)\n    pasos_fin, retornos = [], []\n    n_updates = hp.pasos // (hp.n_envs * hp.n_steps)\n    paso_global, t0 = 0, time.time()\n\n    for upd in range(n_updates):\n        for g in opt.param_groups:                       # lr con decaimiento lineal\n            g["lr"] = hp.lr * (1 - upd / n_updates)\n\n        b_obs, b_act, b_r, b_nt = [], [], [], []\n        for _ in range(hp.n_steps):\n            o_t = torch.as_tensor(obs, dtype=torch.float32, device=device)\n            with torch.no_grad():\n                a = modelo.dist(o_t).sample()\n            a_np = a.cpu().numpy()\n            sig = np.empty_like(obs)\n            r_t = np.zeros(hp.n_envs, dtype=np.float32)\n            nt_t = np.ones(hp.n_envs, dtype=np.float32)\n            for i, e in enumerate(envs):\n                o2, r, term, trunc, _ = e.step(int(a_np[i]))\n                ret_ep[i] += r\n                if trunc and not term:\n                    # Truncamiento por tiempo: el estado no es terminal, se\n                    # bootstrapea con V(s_final) dentro de la recompensa.\n                    with torch.no_grad():\n                        r += hp.gamma * float(modelo.valor(torch.as_tensor(o2, dtype=torch.float32, device=device)))\n                if term or trunc:\n                    nt_t[i] = 0.0\n                    pasos_fin.append(paso_global + i + 1)\n                    retornos.append(ret_ep[i])\n                    ret_ep[i] = 0.0\n                    o2, _ = e.reset()\n                r_t[i] = r\n                sig[i] = o2\n            b_obs.append(o_t); b_act.append(a)\n            b_r.append(torch.as_tensor(r_t, device=device)); b_nt.append(torch.as_tensor(nt_t, device=device))\n            obs = sig\n            paso_global += hp.n_envs\n\n        b_obs = torch.stack(b_obs)                       # [T, N, n_obs]\n        b_act, b_r, b_nt = torch.stack(b_act), torch.stack(b_r), torch.stack(b_nt)\n        dist = modelo.dist(b_obs)\n        v = modelo.valor(b_obs)                           # [T, N]\n        with torch.no_grad():\n            v_ult = modelo.valor(torch.as_tensor(obs, dtype=torch.float32, device=device))\n            adv, ret = ventajas_gae(b_r, v.detach(), v_ult, b_nt, hp.gamma, hp.gae_lambda)\n\n        loss_actor = -(dist.log_prob(b_act) * adv).mean()     # gradiente de política con ventaja\n        loss_critico = F.mse_loss(v, ret)\n        entropia = dist.entropy().mean()\n        loss = loss_actor + hp.vf_coef * loss_critico - hp.ent_coef * entropia\n        opt.zero_grad()\n        loss.backward()\n        nn.utils.clip_grad_norm_(modelo.parameters(), hp.max_grad_norm)\n        opt.step()\n\n        if verbose and (upd + 1) % max(1, n_updates // 10) == 0 and retornos:\n            print(f"  paso {paso_global:7d} | ep {len(retornos):4d} | media últ. 50: "\n                  f"{np.mean(retornos[-50:]):6.1f} | entropía {entropia.item():.2f}")\n    if verbose:\n        print(f"  A2C seed {seed}: {(time.time() - t0) / 60:.1f} min")\n    return modelo, np.array(pasos_fin), np.array(retornos)\n\n\ndef politica_a2c(modelo):\n    """Determinista para evaluar: la acción más probable."""\n    def pi(obs):\n        with torch.no_grad():\n            return int(modelo.actor(torch.as_tensor(obs, dtype=torch.float32, device=device)).argmax().item())\n    return pi\n\n\n# ===========================================================================\n# Referencia no-RL y evaluación\n# ===========================================================================\n\ndef politica_pd(env, kp=4.5, kd=3.0):\n    """PD a mano sobre la observación normalizada, cuantizado al nivel discreto\n    más cercano. No aprende nada: sirve de vara para medir a los agentes."""\n    def pi(obs):\n        u = -(kp * obs[0] + kd * obs[1])\n        return int(np.argmin(np.abs(env.niveles - np.clip(u, -1, 1))))\n    return pi\n\n\ndef evaluar(politica, crear_env, n=200, semilla0=10_000):\n    """Política fija (sin exploración) en episodios con semillas que el\n    entrenamiento no usó."""\n    env = crear_env()\n    retornos, sobrevive, theta_abs, du_abs = [], [], [], []\n    for k in range(n):\n        obs, _ = env.reset(seed=semilla0 + k)\n        total, u_prev, fin = 0.0, 0.0, False\n        while not fin:\n            obs, r, term, trunc, info = env.step(politica(obs))\n            fin = term or trunc\n            total += r\n            theta_abs.append(abs(info["theta"]))\n            du_abs.append(abs(info["u"] - u_prev)); u_prev = info["u"]\n        retornos.append(total)\n        sobrevive.append(not term)\n    return {"retorno": float(np.mean(retornos)), "retorno_std": float(np.std(retornos)),\n            "supervivencia": float(100 * np.mean(sobrevive)),\n            "theta_medio_deg": float(np.degrees(np.mean(theta_abs))),\n            "du_medio": float(np.mean(du_abs))}\n\n\ndef trayectoria(politica, crear_env, theta0_deg=3.0):\n    """θ(t) y u(t) desde una inclinación inicial fija, para graficar."""\n    env = crear_env()\n    obs, _ = env.reset(seed=0, options={"theta0": np.radians(theta0_deg)})\n    t, th, u, fin = [0.0], [theta0_deg], [0.0], False\n    while not fin:\n        obs, _, term, trunc, info = env.step(politica(obs))\n        fin = term or trunc\n        t.append(t[-1] + 0.01); th.append(np.degrees(info["theta"])); u.append(info["u"])\n    return np.array(t), np.array(th), np.array(u)\n')
print('escrito', 'agentes.py')

## 1. El problema en una frase
El robot es un **péndulo invertido**: si está un poco inclinado, la gravedad lo inclina más (es inestable).
La única forma de sostenerlo es mover las ruedas hacia el lado al que cae. Hay que decidir, 100 veces
por segundo, **cuánto y hacia dónde** empujar.

Nadie le dice al agente la ecuación. Solo prueba acciones, recibe una recompensa y aprende.

## El modelo del robot

Péndulo invertido sobre dos ruedas, linealizado alrededor de la vertical:

$$J\,\ddot\theta = m g l\,\theta + \tau_{motor}(u) \quad\Longrightarrow\quad \ddot\theta = \omega_0^2\,\theta + K_U\,k(|u|)\,u$$

| Parámetro | Valor | Origen |
|---|---|---|
| $\omega_0$ | 8.0 rad/s | medido: video del robot colgado del eje (registro §7.22) |
| $J$ | 0.0159 kg·m² | medido: dos apoyos + $J = mgl/\omega_0^2$ (§7.23) |
| $m\,l$ | 0.1040 kg·m | medido: dos apoyos sobre balanza (§7.23) |
| $K_U$ | **0.043** rad/s² por unidad de duty | **supuesto** (ver abajo) |
| zona muerta M1 / M2 | 66 / 73 de duty | medida con el driver viejo, escalada igual que $K_U$ |
| tope de duty | 121 | 6 V del motor / 12.6 V de batería |

**Por qué $K_U$ es un supuesto.** El torque del motor con el driver nuevo (BTS7960) no se llegó a medir:
el ESP32 se dañó antes del ensayo. Se parte del torque medido con balanza usando el driver anterior
(L298N, dos motores a la vez): $4.89\times10^{-4}$ N·m por unidad de duty. El BTS7960 no tiene la caída de
~3 V del L298N, por lo que entrega entre 1.0× y 1.6× ese torque. Se adopta **1.4×** (medianamente optimista):

$$K_U = \frac{1.4 \times 4.89\times10^{-4}}{0.0159} \approx 0.043 \qquad\Rightarrow\qquad \theta_{umbral} = \frac{K_U \cdot 121}{\omega_0^2} \approx 4.7^\circ$$

$\theta_{umbral}$ es el máximo ángulo del que el robot puede recuperarse con todo el torque disponible.


**Lectura intuitiva:** $\omega_0^2\theta$ es la gravedad (empuja a caer, más cuanto más inclinado);
$K_U u$ es el motor (empuja de vuelta). Si el robot pasa de ~4.7°, ni con todo el motor se salva.

In [ ]:
import numpy as np
from entorno import RobotBalanceadorEnv, K_U, THETA_UMBRAL
print(f"K_U = {K_U:.4f}   θ_umbral = {np.degrees(THETA_UMBRAL):.2f}°")

# Sin motor: ¿cuánto tarda en caer desde 1°?
env = RobotBalanceadorEnv()
env.reset(options={"theta0": np.radians(1.0)})
for paso in range(1, 501):
    _, r, cayo, _, info = env.step(4)          # acción 4 = duty 0
    if cayo: break
print(f"Sin hacer nada cae en {paso*10} ms")

## El problema como MDP

| Elemento | Definición |
|---|---|
| Estado $s$ | $(\theta,\ \dot\theta,\ u_{anterior})$, normalizados |
| Acción $a$ | 9 niveles de duty: −121, −105, −90, −75, 0, +75, +90, +105, +121 |
| Transición | la ecuación de arriba integrada a 1 ms; el agente decide cada 10 ms (100 Hz) |
| Recompensa | $r = 1 - (\theta/10^\circ)^2 - 0.05\,(u/121)^2 - 0.05\,(\Delta u/242)^2$ por paso |
| Fin del episodio | cae ($\lvert\theta\rvert > 10^\circ$, *terminated*) o pasan 5 s (*truncated*) |
| Estado inicial | $\lvert\theta_0\rvert \le 3.06^\circ$, $\lvert\dot\theta_0\rvert \le 0.15$ rad/s (error de colocación a mano medido) |
| Descuento | $\gamma = 0.99$ (horizonte ≈ 100 pasos = 1 s) |

El retorno máximo es ≈ 500 (500 pasos con recompensa ≈ 1). Un episodio en que el robot cae
rápido suma ≈ 30.


**Por qué la recompensa tiene esa forma:** el `1` premia seguir de pie; $-(\theta/10°)^2$ premia estar
derecho; los dos últimos términos castigan gastar motor y cambiar bruscamente (en el robot real
eso calienta el driver y hace vibrar el chasis).

**Por qué el estado incluye $u_{anterior}$:** la recompensa depende de $\Delta u$; sin él, el agente
no podría saber cuánto le va a costar cambiar de acción (el estado no sería *markoviano*).

In [ ]:
# Jugá un episodio con una política "a mano" (PD) y uno al azar
from agentes import politica_pd, evaluar
pd = politica_pd(env)
azar = lambda obs: env.action_space.sample()
for nombre, pi in [("PD a mano", pd), ("al azar", azar)]:
    ev = evaluar(pi, RobotBalanceadorEnv, n=20)
    print(f"{nombre:10s} retorno {ev['retorno']:6.1f}   sobrevive {ev['supervivencia']:5.1f}%")

## 2. De Q-learning a DQN

En el TP2 (Q-learning) había una **tabla** $Q(s,a)$ y se actualizaba con

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,]$$

Acá el estado es **continuo** ($\theta$ puede valer 1.2345°): no entra en una tabla.
**DQN** reemplaza la tabla por una **red neuronal** que recibe $s$ (3 números) y devuelve $Q(s,a)$ para
las 9 acciones. La idea de la actualización es la misma; se agregan tres trucos para que sea estable:

| Truco | Qué hace | Por qué |
|---|---|---|
| ε-greedy | explora al azar con probabilidad ε (1 → 0.02) | sin explorar no descubre qué funciona |
| *Replay buffer* | guarda transiciones y entrena con lotes al azar | pasos seguidos se parecen mucho; mezclar evita sesgos |
| Red objetivo | copia congelada de la red para calcular $r+\gamma \max Q(s',a')$ | si el blanco se mueve en cada paso, el aprendizaje oscila |

En el código: `entrenar_dqn` en `agentes.py`. El corazón es `objetivo_td`.

## 3. Double DQN: una línea de diferencia

El $\max$ de DQN **sobreestima** $Q$: si algunas estimaciones tienen error para arriba, el máximo las elige.
Double DQN separa los dos roles:

- DQN: $\;y = r + \gamma\, Q_{obj}(s', \arg\max_{a'} Q_{obj}(s',a'))$ — la misma red elige y evalúa
- Double DQN: $\;y = r + \gamma\, Q_{obj}(s', \arg\max_{a'} Q_{red}(s',a'))$ — una elige, la otra evalúa

In [ ]:
import torch
from agentes import QNet, objetivo_td
torch.manual_seed(0)
red, red_obj = QNet(3, 9), QNet(3, 9)                 # dos redes sin entrenar
s2 = torch.randn(5, 3); r = torch.ones(5, 1); d = torch.zeros(5, 1)
print("objetivo DQN       :", objetivo_td(red, red_obj, r, s2, d, 0.99, double=False).squeeze().numpy().round(3))
print("objetivo Double DQN:", objetivo_td(red, red_obj, r, s2, d, 0.99, double=True).squeeze().numpy().round(3))
print("→ el de Double DQN nunca es mayor: evalúa con red_obj una acción que no eligió red_obj")

## 4. A2C: aprender la política directamente

DQN aprende **valores** y la política sale de elegir el máximo. A2C aprende **dos cosas a la vez**:

- **Actor** $\pi(a|s)$: la probabilidad de cada acción. Es la política.
- **Crítico** $V(s)$: cuánto retorno espera desde $s$.

Después de probar acciones, calcula la **ventaja** $A = (\text{retorno obtenido}) - V(s)$:
"¿me fue mejor o peor de lo que esperaba?". Si $A>0$, sube la probabilidad de esa acción; si $A<0$, la baja.

| | DQN / Double DQN | A2C |
|---|---|---|
| Aprende | $Q(s,a)$ | $\pi(a\mid s)$ y $V(s)$ |
| Datos | *off-policy*: reutiliza experiencia vieja (replay) | *on-policy*: usa solo lo recién jugado |
| Explora con | ε-greedy | el azar propio de $\pi$ (+ bonus de entropía) |
| Para decorrelacionar | replay buffer | 8 entornos en paralelo |

En el código: `entrenar_a2c` y `ventajas_gae` en `agentes.py` (GAE es una forma suavizada de calcular $A$).

In [ ]:
# Entrenamiento corto de A2C (≈ 20 s) para verlo aprender
from agentes import entrenar_a2c, HiperA2C, politica_a2c
modelo, pasos, retornos = entrenar_a2c(RobotBalanceadorEnv, HiperA2C(pasos=150_000), seed=0)
print(evaluar(politica_a2c(modelo), RobotBalanceadorEnv, n=20))

## 5. Cómo leer los resultados

- **Curva de convergencia**: retorno por episodio (media móvil) a lo largo del entrenamiento.
  Empieza en ≈ 30 (cae enseguida) y sube hacia 500 (se sostiene los 5 s).
- **Contra episodios vs contra pasos**: A2C juega muchos episodios cortos al principio (8 entornos), así que
  "por episodio" lo favorece visualmente. **Contra pasos** es la comparación justa de eficiencia.
- **Evaluación sin exploración**: lo que importa al final; se mide en episodios nuevos, con ε = 0 (DQN)
  o eligiendo la acción más probable (A2C).
- **Varias semillas**: el mismo algoritmo puede salir mejor o peor según el azar. Por eso 3 semillas.
- **Límite del modelo**: todo esto vale para el robot *simulado* con $K_U$ supuesto. En el robot real
  faltaría validar $K_U$ y considerar ruido del sensor, demora y las ruedas.

## Referencias

1. Sutton, R. S. y Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2.ª ed.). MIT Press. Cap. 6 (Q-learning), 9 (aproximación de funciones) y 13 (actor-crítico).
2. Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature* 518, 529–533. (DQN)
3. van Hasselt, H., Guez, A. y Silver, D. (2016). Deep Reinforcement Learning with Double Q-learning. *AAAI*. (Double DQN)
4. Mnih, V. et al. (2016). Asynchronous Methods for Deep Reinforcement Learning. *ICML*. (A3C/A2C)
5. Schulman, J. et al. (2016). High-Dimensional Continuous Control Using Generalized Advantage Estimation. *ICLR*. (GAE)
6. Towers, M. et al. (2024). Gymnasium: A Standard Interface for Reinforcement Learning Environments. arXiv:2407.17032.
7. Material de la materia (FIUBA, 2026): clase 5 (DQN), clase 7 (A2C), TP3.
8. Registro de pruebas del robot: `docs/conexiones-registro-pruebas.md` del repositorio (§7.20, §7.22, §7.23, §8.5).